# Prepare Training Dataset for Granite Fine-tuning

This notebook generates instruction-tuning dataset from employee activity logs for cybersecurity analysis.

## 1. Install Dependencies

In [ ]:
!pip install pandas numpy

## 2. Import Libraries

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict
from pathlib import Path

print("✓ Libraries imported")

## 3. Load Raw Activity Logs

In [ ]:
INPUT_FILE = '../datasets/employee_activity_logs.json'
OUTPUT_FILE = '../datasets/security_training_data.jsonl'

print(f"Loading logs from: {INPUT_FILE}")

with open(INPUT_FILE, 'r') as f:
    logs = json.load(f)

df = pd.DataFrame(logs)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"\nLoaded {len(df)} log entries")
print(f"Unique employees: {df['employee_id'].nunique()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

print("\nSample logs:")
df.head()

## 4. Analyze Employee Behavior Patterns

In [ ]:
def analyze_employee_behavior(logs_df, employee_id):
    """
    Analyze complete behavior pattern for an employee.
    """
    emp_logs = logs_df[logs_df['employee_id'] == employee_id].sort_values('timestamp')
    
    if len(emp_logs) == 0:
        return None, None
    
    analysis = {
        'employee_id': employee_id,
        'total_sessions': len(emp_logs),
        'unique_ips': emp_logs['ip_address'].nunique(),
        'unique_locations': emp_logs['geo_location'].nunique() if 'geo_location' in emp_logs else 1,
        'actions': emp_logs['action_type'].value_counts().to_dict(),
        'night_accesses': 0,
        'weekend_accesses': 0,
        'failed_logins': len(emp_logs[emp_logs['action_type'] == 'failed_login']),
        'sensitive_resources': [],
        'external_ips': [],
        'suspicious_patterns': []
    }
    
    # Analyze each log entry
    for _, log in emp_logs.iterrows():
        dt = log['timestamp']
        hour = dt.hour
        
        # Check time patterns
        if hour < 6 or hour > 22:
            analysis['night_accesses'] += 1
        
        if dt.weekday() >= 5:
            analysis['weekend_accesses'] += 1
        
        # Check IP patterns
        ip = log['ip_address']
        if not ip.startswith(('192.168', '10.')):
            if ip not in analysis['external_ips']:
                analysis['external_ips'].append(ip)
        
        # Check resource access
        resource = log.get('resource_accessed', '')
        if pd.notna(resource):
            sensitive_keywords = ['confidential', 'salary', 'database', 'backup', 'layoff', 'secret']
            if any(word in str(resource).lower() for word in sensitive_keywords):
                analysis['sensitive_resources'].append(resource)
    
    # Identify suspicious patterns
    if analysis['external_ips']:
        analysis['suspicious_patterns'].append(
            f"Access from {len(analysis['external_ips'])} external IP address(es)"
        )
    
    if analysis['night_accesses'] > 2:
        analysis['suspicious_patterns'].append(
            f"{analysis['night_accesses']} after-hours access events"
        )
    
    if analysis['failed_logins'] > 3:
        analysis['suspicious_patterns'].append(
            f"{analysis['failed_logins']} failed login attempts (possible compromise)"
        )
    
    if len(analysis['sensitive_resources']) > 2:
        analysis['suspicious_patterns'].append(
            f"Multiple accesses to sensitive data ({len(analysis['sensitive_resources'])} files)"
        )
    
    if analysis['unique_ips'] > 4:
        analysis['suspicious_patterns'].append(
            f"Unusual number of IP addresses ({analysis['unique_ips']})"
        )
    
    return analysis, emp_logs

print("✓ Analysis function defined")

## 5. Create Activity Summary

In [ ]:
def create_activity_summary(emp_logs, max_events=10):
    """
    Create timeline of recent activities.
    """
    activities = []
    
    for _, log in emp_logs.tail(max_events).iterrows():
        dt = log['timestamp']
        activity = f"{dt.strftime('%Y-%m-%d %H:%M')}: {log['action_type']}"
        
        if pd.notna(log.get('resource_accessed')):
            activity += f" - {log['resource_accessed']}"
        
        activity += f" from {log['ip_address']}"
        
        if pd.notna(log.get('geo_location')):
            activity += f" ({log['geo_location']})"
        
        activities.append(activity)
    
    return activities

print("✓ Summary function defined")

## 6. Generate Training Examples

In [ ]:
def generate_training_example(analysis, emp_logs):
    """
    Generate instruction-tuning example from employee analysis.
    """
    system_prompt = (
        "You are a cybersecurity expert specializing in employee behavior analytics "
        "and insider threat detection. Analyze employee activity logs and provide "
        "comprehensive security assessments with risk levels and actionable recommendations."
    )
    
    activities = create_activity_summary(emp_logs)
    
    # Create user prompt
    user_prompt = f"""Analyze employee behavior for: {analysis['employee_id']}

Activity Summary (30-day period):
- Total login sessions: {analysis['total_sessions']}
- Unique IP addresses used: {analysis['unique_ips']}
- Different locations: {analysis['unique_locations']}
- After-hours accesses: {analysis['night_accesses']}
- Weekend accesses: {analysis['weekend_accesses']}
- Failed login attempts: {analysis['failed_logins']}

Recent Activity Timeline:
{chr(10).join(f'  {a}' for a in activities)}

Action Distribution:
{json.dumps(analysis['actions'], indent=2)}

External IPs detected: {len(analysis['external_ips'])}
Sensitive resources accessed: {len(analysis['sensitive_resources'])}

Provide detailed security assessment including risk level, behavioral analysis, and recommendations."""
    
    # Determine if suspicious
    is_suspicious = len(analysis['suspicious_patterns']) > 0
    
    # Generate assistant response
    if is_suspicious:
        risk_level = (
            'CRITICAL' if len(analysis['suspicious_patterns']) >= 3
            else 'HIGH' if len(analysis['suspicious_patterns']) >= 2
            else 'MEDIUM'
        )
        
        assistant_response = f"""SECURITY ASSESSMENT - ANOMALOUS BEHAVIOR DETECTED

Employee ID: {analysis['employee_id']}
Risk Level: {risk_level}

BEHAVIORAL ANALYSIS:
Over the monitored period of {analysis['total_sessions']} sessions, this employee has exhibited multiple concerning patterns that deviate from normal baseline behavior.

SUSPICIOUS INDICATORS IDENTIFIED:
{chr(10).join(f'  -  {p}' for p in analysis['suspicious_patterns'])}

DETAILED FINDINGS:

1. Access Patterns:
   - Unusual time-based activity detected
   - Multiple IP addresses suggest possible location changes or VPN usage
   - Access from {analysis['unique_locations']} different location(s)

2. Resource Access:
   - {len(analysis['sensitive_resources'])} sensitive/confidential resource access event(s)
   - Pattern suggests potential data gathering or exfiltration preparation

3. Authentication Anomalies:
   - {analysis['failed_logins']} failed login attempt(s)
   - Unusual authentication patterns compared to user baseline

RISK ASSESSMENT:
The combination of after-hours access, external IP usage, and sensitive data access creates a high-risk profile. This behavior is consistent with:
- Potential insider threat activity
- Possible account compromise
- Data exfiltration preparation
- Unauthorized access attempts

IMMEDIATE RECOMMENDATIONS:
1. PRIORITY: Enable enhanced real-time monitoring for this account
2. Review all recent data downloads and file transfers
3. Verify legitimacy of all access events with employee
4. Implement additional authentication requirements (MFA)
5. Restrict access to highly sensitive systems temporarily
6. Coordinate with HR for behavioral assessment
7. Document all findings in incident management system

FOLLOW-UP ACTIONS:
- Schedule security interview with employee within 24 hours
- Audit all systems accessed in last 30 days
- Review email and file sharing activities
- Monitor for any data exfiltration indicators
- Preserve all logs for potential forensic investigation

TIMELINE FOR ACTION:
- Immediate (0-4 hours): Enable enhanced monitoring
- Short-term (4-24 hours): Conduct employee verification
- Medium-term (1-7 days): Complete full security audit
- Long-term: Reassess risk profile after investigation"""
    
    else:
        assistant_response = f"""SECURITY ASSESSMENT - NORMAL BEHAVIOR PATTERN

Employee ID: {analysis['employee_id']}
Risk Level: LOW

BEHAVIORAL ANALYSIS:
Based on {analysis['total_sessions']} login sessions over the monitoring period, this employee demonstrates consistent and appropriate access patterns aligned with their role and responsibilities.

POSITIVE INDICATORS:
  -  All access from authorized IP ranges (internal network)
  -  Activity concentrated during standard business hours
  -  No suspicious authentication anomalies
  -  Consistent location patterns
  -  Appropriate resource access for role
  -  Stable behavioral baseline maintained

ACTIVITY PROFILE:
- IP Address Usage: {analysis['unique_ips']} address(es) (normal variation)
- Location Consistency: {analysis['unique_locations']} location(s) (expected)
- Time-based Access: Primary activity during business hours
- Authentication: Clean record, no failed attempts
- Resource Access: Aligned with job function

SECURITY STATUS:
No concerning patterns or anomalies detected. The employee's behavior is consistent with normal work activities and shows no indicators of:
- Insider threat activity
- Account compromise
- Unauthorized access attempts
- Data exfiltration preparation

RECOMMENDATIONS:
1. Continue standard monitoring procedures
2. No immediate action required
3. Maintain current access levels
4. Include in routine periodic reviews

MONITORING STATUS:
- Current Risk Level: LOW
- Next Review: Scheduled routine assessment
- Access Status: Approved and appropriate
- No alerts or concerns flagged"""
    
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_response}
        ]
    }

print("✓ Training example generator defined")

## 7. Process All Employees

In [ ]:
print("Processing all employees...\n")

training_examples = []
employee_stats = {
    'total': 0,
    'suspicious': 0,
    'normal': 0
}

for employee_id in df['employee_id'].unique():
    analysis, emp_logs = analyze_employee_behavior(df, employee_id)
    
    if analysis is None:
        continue
    
    example = generate_training_example(analysis, emp_logs)
    training_examples.append(example)
    
    employee_stats['total'] += 1
    
    if len(analysis['suspicious_patterns']) > 0:
        employee_stats['suspicious'] += 1
        print(f"✗ {employee_id}: SUSPICIOUS ({len(analysis['suspicious_patterns'])} indicators)")
    else:
        employee_stats['normal'] += 1
        print(f"✓ {employee_id}: Normal")

print(f"\n{'='*60}")
print("DATASET GENERATION COMPLETE")
print(f"{'='*60}")
print(f"Total examples generated: {len(training_examples)}")
print(f"  - Suspicious behavior: {employee_stats['suspicious']}")
print(f"  - Normal behavior: {employee_stats['normal']}")
print(f"  - Balance: {employee_stats['suspicious']/len(training_examples)*100:.1f}% anomalous")

## 8. Save Training Dataset

In [ ]:
print(f"Saving training data to: {OUTPUT_FILE}")

with open(OUTPUT_FILE, 'w') as f:
    for example in training_examples:
        f.write(json.dumps(example) + '\n')

print(f"\n✓ Saved {len(training_examples)} training examples")
print(f"\nFile size: {Path(OUTPUT_FILE).stat().st_size / 1024:.2f} KB")

## 9. Preview Training Examples

In [ ]:


print("SYSTEM:")
print(sample['messages']['content'][:200] + "...\n")

print("USER:")
print(sample['messages']['content'][:300] + "...\n")

print("ASSISTANT:")
print(sample['messages']['content'][:500] + "...")

print("\n" + "="*60)

## 10. Validate Dataset

In [ ]:
print("Validating dataset...\n")

# Check all examples have required fields
for i, example in enumerate(training_examples):
    assert 'messages' in example, f"Example {i} missing 'messages'"
    assert len(example['messages']) == 3, f"Example {i} has {len(example['messages'])} messages, expected 3"
    
    for j, msg in enumerate(example['messages']):
        assert 'role' in msg, f"Example {i}, message {j} missing 'role'"
        assert 'content' in msg, f"Example {i}, message {j} missing 'content'"
        assert len(msg['content']) > 0, f"Example {i}, message {j} has empty content"

# Calculate average lengths
system_lengths = [len(ex['messages']['content']) for ex in training_examples]
user_lengths = [len(ex['messages']['content']) for ex in training_examples]
assistant_lengths = [len(ex['messages']['content']) for ex in training_examples]

print("✓ All examples valid!\n")
print("Average Content Lengths:")
print(f"  System prompts: {np.mean(system_lengths):.0f} chars")
print(f"  User prompts: {np.mean(user_lengths):.0f} chars")
print(f"  Assistant responses: {np.mean(assistant_lengths):.0f} chars")

total_tokens_estimate = sum(system_lengths + user_lengths + assistant_lengths) / 4  # rough estimate
print(f"\nEstimated total tokens: ~{total_tokens_estimate:,.0f}")

## Summary

✅ Dataset preparation complete!

**Output:** `../datasets/security_training_data.jsonl`

**Ready for:** Fine-tuning Granite LLM in notebook `2_finetune_granite.ipynb`
